# Simulation simplifiée : solaire, diesel et batterie

Ce carnet contient uniquement les **premiers calculs** du cas d'étude d'Abuja (2025). Le fichier CSV de résultats et les tableaux de bord proviennent du dossier de référence fourni et ne sont **pas intégralement régénérés** par ce carnet.

Paramètres du jeu de résultats fourni : puissance photovoltaïque de **8 kW** et demande journalière de **35 kWh**. Ce calcul simple ne prend pas encore en compte les pertes réelles du système.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Fonctionne depuis la racine du projet ou depuis le dossier calculs.
dossier_courant = Path.cwd()
racine_projet = dossier_courant if (dossier_courant / "donnees").exists() else dossier_courant.parent
chemin_donnees = racine_projet / "donnees" / "ensoleillement_abuja_2025.csv"
assert chemin_donnees.is_file(), f"Fichier introuvable : {chemin_donnees}"

# Fichier NASA POWER : 9 lignes de métadonnées avant l'en-tête CSV.
ensoleillement = pd.read_csv(chemin_donnees, skiprows=9)
ensoleillement.head()


In [ ]:
# Hypothèses du cas de référence (simulation simplifiée).
puissance_pv_kw = 8
demande_kwh_jour = 35

# ALLSKY_SFC_SW_DWN correspond à une énergie solaire incidente quotidienne (kWh/m²/j).
# Approximation : production PV = irradiation * puissance nominale (sans pertes).
ensoleillement["Solar_Output"] = ensoleillement["ALLSKY_SFC_SW_DWN"] * puissance_pv_kw
ensoleillement["Load_kWh"] = demande_kwh_jour
ensoleillement["Energy_Gap"] = (ensoleillement["Load_kWh"] - ensoleillement["Solar_Output"]).clip(lower=0)

ensoleillement[["YEAR", "MO", "DY", "Solar_Output", "Load_kWh", "Energy_Gap"]].head()


In [ ]:
# Comparaison des premières valeurs avec le fichier de résultats fourni.
reference = pd.read_csv(racine_projet / "resultats" / "simulation_systeme_hybride.csv")
colonnes = ["Solar_Output", "Load_kWh", "Energy_Gap"]
for nom_colonne in colonnes:
    ecart_max = (ensoleillement[nom_colonne] - reference[nom_colonne]).abs().max()
    print(f"{nom_colonne} : écart maximal = {ecart_max:.6f}")

print("Les colonnes batterie, diesel, coûts et CO2 exigent un modèle complémentaire.")
